# Exploratory Data Analysis: Tree Model Data

**Objective**: This notebook performs a comprehensive exploratory analysis of the preprocessed irrigation dataset specifically curated for tree algorithms (e.g., Decision Tree, Random Forest, etc). The primary goal is to validate data quality and determine the statistical transformations required to optimize tree model performance.

Key Analytical Pillars:
- **Data Health Assessment**: Verification of data integrity, including completeness (missing value analysis) and uniqueness (duplicate checks).
- **Statistical Profiling**: Evaluation of distribution properties—specifically skewness and kurtosis—to decide between standard scaling and robust transformations.
- **Feature Engineering Validation**: Assessment of the current encoding schemes to ensure categorical variables are correctly prepared for distance-based and gradient-based learners.

In [ ]:
# Install required library
!uv pip install --quiet "setuptools<80" fg-data-profiling

### 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

# Get the path to 'irrigation-prediction-system' (the project root)
root_path: Path = Path.cwd().parent.parent.parent
sys.path.append(str(root_path))

### 2. Import Libraries

In [ ]:
import pandas as pd
from data_profiling import ProfileReport
from prefect.settings import PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW, temporary_settings

from src.configs.settings import Settings, get_settings
from src.pipeline.ingestion import load_dataset

%matplotlib inline

### 3. Setup Configs

In [ ]:
settings: Settings = get_settings()

# EDA report title
report_title: str = "Compherensive EDA Report: Linear Data"

# Dataset path
dataset_filepath: Path = settings.EXPERIMENTS_DATA_DIR / settings.TREE_FILENAME

### 4. Load the Dataset

In [ ]:
try:
    with temporary_settings(updates={PREFECT_LOGGING_TO_API_WHEN_MISSING_FLOW: "ignore"}):
        df: pd.DataFrame = load_dataset(dataset_filepath)
except Exception as e:
    sys.stderr.write(f"Ingestion failed: {e}\n")

### 5. Generate EDA Report: Raw Data 

In [ ]:
profile = ProfileReport(df, title=report_title, explorative=True)
profile.to_notebook_iframe()

In [ ]:
num_cols = df.select_dtypes(include=["float"]).columns
print("Numerical Columns:", len(num_cols))

In [ ]:
# Numerical dataframe
num_df: pd.DataFrame = df[num_cols].copy()

# Statistical info
summary: list = []
for col in num_cols:
    clean = df[col].dropna()
    summary.append(
        {
            "Column": col,
            "Count": len(clean),
            "Mean": clean.mean(),
            "Std": clean.std(),
            "Min": clean.min(),
            "Max": clean.max(),
            "Skewness": clean.skew(),
            "Kurtosis": clean.kurtosis(),
            "Q1": clean.quantile(0.25),
            "Q3": clean.quantile(0.75),
            "IQR": clean.quantile(0.75) - clean.quantile(0.25),
        }
    )

pd.DataFrame(summary).sort_values("Skewness", ascending=False)

### 7. EDA Insights and Action Plan

- There are 20 columns and 6,30,000 rows.
- No missing cells or duplicate rows.
- There are 8 categorical variables, and 12 numeric variables.
- All numerical features exhibit skewness values between -0.12 and +0.11. This near-perfect symmetry indicates that heavy-tailed transformations (like Log or Power transforms) are not required.
- Kurtosis values consistently range from -0.94 to -1.26. These negative kurtosis values suggest "platykurtic" distributions with thinner tails than a normal distribution, confirming a lack of extreme outliers that would otherwise bias linear coefficients.